# Machine Learning Model Training: Alzheimer's Disease Detection

This notebook implements and trains multiple machine learning models to predict Alzheimer's disease from MRI biomarker data.

## Approach
- Multiple algorithm comparison
- Hyperparameter optimization via cross-validation
- Comprehensive model evaluation
- Feature importance analysis


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_curve, auc, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("muted")
sys.path.append('../../utils')  # For Kaggle downloader

# Import Kaggle downloader
try:
    from kaggle_downloader import setup_project_data, check_kaggle_credentials
    KAGGLE_AVAILABLE = True
except ImportError:
    KAGGLE_AVAILABLE = False
    print("⚠️ Kaggle downloader not available - install kaggle package")



## 1. Data Preparation


In [ ]:
# Load and preprocess data
df = pd.read_csv('../data/oasis_longitudinal.csv')

# Preprocessing steps
df = df[df['Visit'] == 1].reset_index(drop=True)
df['M/F'] = df['M/F'].map({'F': 0, 'M': 1})
df['Group'] = df['Group'].replace(['Converted'], 'Demented')
df['Group'] = df['Group'].map({'Demented': 1, 'Nondemented': 0})
df['SES'] = df['SES'].fillna(df.groupby('EDUC')['SES'].transform('median'))
df = df.drop(['MRI ID', 'Visit', 'Hand'], axis=1)

# Prepare features and target
features = ['M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
X = df[features].values
y = df['Group'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}")
print(f"Class distribution - Test: {np.bincount(y_test)}")


## 2. Model Training and Evaluation Framework


In [ ]:
# Store results for comparison
results = []
k_folds = 5

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Train and evaluate a model, returning metrics."""
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=k_folds, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Test predictions
    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    test_recall = recall_score(y_test, y_pred, pos_label=1)
    
    # ROC curve
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=1)
        roc_auc = auc(fpr, tpr)
    else:
        fpr, tpr = None, None
        roc_auc = 0.0
    
    return {
        'model_name': model_name,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'test_accuracy': test_acc,
        'test_recall': test_recall,
        'test_auc': roc_auc,
        'model': model,
        'fpr': fpr,
        'tpr': tpr
    }


## 3. Train Multiple Models


In [ ]:
# 3.1 Logistic Regression with hyperparameter tuning
print("Training Logistic Regression...")
best_c = None
best_cv_score = 0

for c in [0.001, 0.1, 1, 10, 100]:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=42)
    cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=k_folds, scoring='accuracy')
    if cv_scores.mean() > best_cv_score:
        best_cv_score = cv_scores.mean()
        best_c = c

lr_model = LogisticRegression(C=best_c, max_iter=1000, random_state=42)
lr_result = evaluate_model(lr_model, X_train_scaled, y_train, X_test_scaled, y_test, "Logistic Regression")
results.append(lr_result)
print(f"✓ Best C: {best_c}, CV Score: {best_cv_score:.4f}")


In [ ]:
# 3.2 Random Forest (typically best performer)
print("Training Random Forest...")
best_params = {}
best_score = 0

for n_est in [50, 100, 200]:
    for max_depth in [5, 10, 15, None]:
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_depth, random_state=42, n_jobs=-1)
        cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=k_folds, scoring='accuracy')
        if cv_scores.mean() > best_score:
            best_score = cv_scores.mean()
            best_params = {'n_estimators': n_est, 'max_depth': max_depth}

rf_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
rf_result = evaluate_model(rf_model, X_train_scaled, y_train, X_test_scaled, y_test, "Random Forest")
results.append(rf_result)
print(f"✓ Best params: {best_params}, CV Score: {best_score:.4f}")


In [ ]:
# 3.3 Support Vector Machine
print("Training SVM...")
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svm_result = evaluate_model(svm_model, X_train_scaled, y_train, X_test_scaled, y_test, "SVM")
results.append(svm_result)
print("✓ SVM training complete")


In [ ]:
# 3.4 Decision Tree
print("Training Decision Tree...")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_result = evaluate_model(dt_model, X_train_scaled, y_train, X_test_scaled, y_test, "Decision Tree")
results.append(dt_result)
print("✓ Decision Tree training complete")


In [ ]:
# 3.5 AdaBoost
print("Training AdaBoost...")
ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada_result = evaluate_model(ada_model, X_train_scaled, y_train, X_test_scaled, y_test, "AdaBoost")
results.append(ada_result)
print("✓ AdaBoost training complete")


## 4. Results Comparison


In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'Model': [r['model_name'] for r in results],
    'CV Accuracy': [f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}" for r in results],
    'Test Accuracy': [f"{r['test_accuracy']:.4f}" for r in results],
    'Test Recall': [f"{r['test_recall']:.4f}" for r in results],
    'Test AUC': [f"{r['test_auc']:.4f}" for r in results]
})

print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)
print(results_df.to_string(index=False))

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['test_accuracy', 'test_recall', 'test_auc']
metric_names = ['Accuracy', 'Recall', 'AUC']

for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
    values = [r[metric] for r in results]
    model_names = [r['model_name'] for r in results]
    
    axes[idx].barh(model_names, values, color=sns.color_palette("husl", len(results)))
    axes[idx].set_title(f'{name} Comparison', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(name, fontsize=10)
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Find best model
best_model_idx = np.argmax([r['test_accuracy'] for r in results])
best_model = results[best_model_idx]
print(f"\n🏆 Best Model: {best_model['model_name']} (Accuracy: {best_model['test_accuracy']:.4f})")


## 5. ROC Curves


In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

for result in results:
    if result['fpr'] is not None and result['tpr'] is not None:
        plt.plot(result['fpr'], result['tpr'], 
                label=f"{result['model_name']} (AUC = {result['test_auc']:.3f})",
                linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Feature Importance Analysis


In [ ]:
# Extract feature importances from tree-based models
importance_data = {}

for result in results:
    model = result['model']
    if hasattr(model, 'feature_importances_'):
        importance_data[result['model_name']] = model.feature_importances_

if importance_data:
    importance_df = pd.DataFrame(importance_data, index=features)
    importance_df = importance_df.sort_values(by=importance_df.columns[0], ascending=False)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    importance_df.plot(kind='barh', ax=ax, width=0.8)
    ax.set_title('Feature Importance Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_ylabel('Features', fontsize=12)
    ax.legend(title='Models', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("\nFeature Importance Rankings:")
    print(importance_df.round(4))


## 7. Save Best Model


In [ ]:
# Save best model and scaler
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model['model'], f)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"✓ Saved best model: {best_model['model_name']}")
print(f"✓ Model saved to: ../models/best_model.pkl")
print(f"✓ Scaler saved to: ../models/scaler.pkl")
